In [1]:
"""
CELL 1 — TASK 1: Raw feature audit (single representative file)
"""

import pandas as pd
import numpy as np
from pathlib import Path

DATASET_ROOT = Path("/kaggle/input/datasets/am2008/edaic-full-data/edaic_btp_dataset")
RAW_DIR = DATASET_ROOT / "raw_egemaps"
LABELS_DIR = DATASET_ROOT / "labels"

train_split = pd.read_csv(LABELS_DIR / "train_split.csv")
sample_pid = train_split["Participant_ID"].iloc[0]
sample_path = list((RAW_DIR / f"{sample_pid}_P").glob("*egemaps.csv"))[0]

df_sample = pd.read_csv(sample_path, sep=";")

print(f"Sample file: {sample_path}")
print(f"\nAll columns ({len(df_sample.columns)}):")
print(list(df_sample.columns))

feature_cols = [c for c in df_sample.columns if c not in ("name", "frameTime")]
print(f"\nColumns after removing 'name' and 'frameTime' ({len(feature_cols)}):")
for i, c in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {c}")

print("\nDtypes:")
print(df_sample[feature_cols].dtypes)

missing_pct = df_sample[feature_cols].isna().mean() * 100
inf_pct = df_sample[feature_cols].apply(lambda col: np.isinf(col).mean() * 100)
print("\nMissing value % per column:")
print(missing_pct[missing_pct > 0] if missing_pct.sum() > 0 else "  None — 0% missing across all columns")
print("\nInfinite value % per column:")
print(inf_pct[inf_pct > 0] if inf_pct.sum() > 0 else "  None — 0% infinite across all columns")

# ------------------------------------------------------------------
# Determine what these 23 features actually are
# ------------------------------------------------------------------
print("\n" + "=" * 70)
print("FEATURE SET IDENTIFICATION")
print("=" * 70)
mfcc_feats = [c for c in feature_cols if "mfcc" in c.lower()]
egemaps_llds = [c for c in feature_cols if "mfcc" not in c.lower()]
has_functionals = any(c.endswith(("_mean", "_std", "_amean", "_stddev")) for c in feature_cols)

print(f"MFCC-named columns: {len(mfcc_feats)} -> {mfcc_feats}")
print(f"Remaining (non-MFCC) columns: {len(egemaps_llds)} -> {egemaps_llds}")
print(f"\nColumn naming ends in '_sma3'/'_sma3nz' (smoothed LLD, not functional-summary suffix like '_amean'): "
      f"{all(c.endswith(('_sma3', '_sma3nz')) for c in feature_cols)}")
print(f"Row count per file (frame-level, not one-row-per-participant functionals): {len(df_sample)} rows")
print("""
CONCLUSION: These are LOW-LEVEL DESCRIPTORS (LLDs), not eGeMAPS functionals.
- 19 columns are the standard eGeMAPS v01b LLD set (Loudness, alphaRatio,
  hammarbergIndex, slopes, spectralFlux, F0/jitter/shimmer/HNR, formants).
- 4 columns (mfcc1-4_sma3) are MFCCs APPENDED to the eGeMAPS LLD set —
  this is NOT standard eGeMAPS (which excludes MFCCs). This matches
  openSMILE's "eGeMAPSv01b + MFCC" extended low-level-descriptor config,
  a common OpenSMILE combination but not the eGeMAPS functional feature
  set (which is 88-dim summary stats, not 23 raw per-frame streams).
- Since these are per-frame LLDs (one row per ~10ms frame), the 7-stat
  aggregation pipeline is manually computing functionals FROM LLDs —
  a valid, standard approach, but distinct from openSMILE's own
  built-in eGeMAPS functional extractor (different formula details
  may not match exactly if compared to published eGeMAPS papers).
""")

Sample file: /kaggle/input/datasets/am2008/edaic-full-data/edaic_btp_dataset/raw_egemaps/302_P/302_OpenSMILE2.3.0_egemaps.csv

All columns (25):
['name', 'frameTime', 'Loudness_sma3', 'alphaRatio_sma3', 'hammarbergIndex_sma3', 'slope0-500_sma3', 'slope500-1500_sma3', 'spectralFlux_sma3', 'mfcc1_sma3', 'mfcc2_sma3', 'mfcc3_sma3', 'mfcc4_sma3', 'F0semitoneFrom27.5Hz_sma3nz', 'jitterLocal_sma3nz', 'shimmerLocaldB_sma3nz', 'HNRdBACF_sma3nz', 'logRelF0-H1-H2_sma3nz', 'logRelF0-H1-A3_sma3nz', 'F1frequency_sma3nz', 'F1bandwidth_sma3nz', 'F1amplitudeLogRelF0_sma3nz', 'F2frequency_sma3nz', 'F2amplitudeLogRelF0_sma3nz', 'F3frequency_sma3nz', 'F3amplitudeLogRelF0_sma3nz']

Columns after removing 'name' and 'frameTime' (23):
   1. Loudness_sma3
   2. alphaRatio_sma3
   3. hammarbergIndex_sma3
   4. slope0-500_sma3
   5. slope500-1500_sma3
   6. spectralFlux_sma3
   7. mfcc1_sma3
   8. mfcc2_sma3
   9. mfcc3_sma3
  10. mfcc4_sma3
  11. F0semitoneFrom27.5Hz_sma3nz
  12. jitterLocal_sma3nz
  13. shim

In [2]:
"""
CELL 2 — TASK 2: Consistency check across ALL 275 raw files
"""

all_ids = []
for split in ["train", "dev", "test"]:
    d = pd.read_csv(LABELS_DIR / f"{split}_split.csv")
    all_ids.extend(d["Participant_ID"].tolist())

print(f"Checking {len(all_ids)} participants...")

reference_cols = None
exceptions = []

for pid in all_ids:
    matches = list((RAW_DIR / f"{pid}_P").glob("*egemaps.csv"))
    if len(matches) == 0:
        exceptions.append((pid, "FILE NOT FOUND"))
        continue
    if len(matches) > 1:
        exceptions.append((pid, f"MULTIPLE MATCHES: {[m.name for m in matches]}"))
    # check for BoAW contamination
    boaw_present = any("boaw" in m.name.lower() for m in matches)
    if boaw_present and len(matches) == 1:
        exceptions.append((pid, f"Matched file appears to be BoAW: {matches[0].name}"))

    try:
        cols = pd.read_csv(matches[0], sep=";", nrows=0).columns.tolist()
        feat_cols = [c for c in cols if c not in ("name", "frameTime")]
    except Exception as e:
        exceptions.append((pid, f"READ ERROR: {e}"))
        continue

    if reference_cols is None:
        reference_cols = feat_cols
    elif feat_cols != reference_cols:
        exceptions.append((pid, f"COLUMN MISMATCH: {feat_cols}"))
    if len(feat_cols) != 23:
        exceptions.append((pid, f"WRONG COLUMN COUNT: {len(feat_cols)}"))

print(f"\nChecked {len(all_ids)} participants.")
if exceptions:
    print(f"\n{len(exceptions)} EXCEPTIONS FOUND:")
    for pid, issue in exceptions:
        print(f"  {pid}: {issue}")
else:
    print("\nAll 275 participants have identical 23 acoustic feature columns.")

Checking 275 participants...

Checked 275 participants.

All 275 participants have identical 23 acoustic feature columns.


In [3]:
"""
CELL 3 — Rebuild the 161-dim feature set (reused for all downstream tasks)
"""

STATS = ["mean", "std", "median", "min", "max", "p25", "p75"]

def compute_participant_stats(arr):
    arr = np.where(np.isfinite(arr), arr, np.nan)
    stats = [
        np.nanmean(arr, axis=0), np.nanstd(arr, axis=0), np.nanmedian(arr, axis=0),
        np.nanmin(arr, axis=0), np.nanmax(arr, axis=0),
        np.nanpercentile(arr, 25, axis=0), np.nanpercentile(arr, 75, axis=0),
    ]
    vec = np.concatenate(stats)
    return np.where(np.isfinite(vec), vec, 0.0)

def load_split(split_name):
    labels = pd.read_csv(LABELS_DIR / f"{split_name}_split.csv")[["Participant_ID", "PHQ_Binary"]]
    rows, kept_pids, kept_labels = [], [], []
    feature_cols_local = None
    for _, row in labels.iterrows():
        pid, y = row["Participant_ID"], row["PHQ_Binary"]
        matches = list((RAW_DIR / f"{pid}_P").glob("*egemaps.csv"))
        matches = [m for m in matches if "boaw" not in m.name.lower()]
        if not matches:
            print(f"[warn] missing {pid} ({split_name})")
            continue
        d = pd.read_csv(matches[0], sep=";")
        if feature_cols_local is None:
            feature_cols_local = [c for c in d.columns if c not in ("name", "frameTime")]
        arr = d[feature_cols_local].values.astype(np.float64)
        rows.append(compute_participant_stats(arr))
        kept_pids.append(pid); kept_labels.append(y)
    feat_names = [f"{fc}_{s}" for s in STATS for fc in feature_cols_local]
    X = pd.DataFrame(rows, columns=feat_names)
    X.insert(0, "Participant_ID", kept_pids)
    X["PHQ_Binary"] = kept_labels
    return X

train_data = load_split("train")
dev_data = load_split("dev")
test_data = load_split("test")
feature_cols = [c for c in train_data.columns if c not in ("Participant_ID", "PHQ_Binary")]
print(f"Train/Dev/Test: {len(train_data)}/{len(dev_data)}/{len(test_data)} | features: {len(feature_cols)}")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

X_train_raw = train_data[feature_cols].values
X_dev_raw = dev_data[feature_cols].values
X_test_raw = test_data[feature_cols].values
y_train = train_data["PHQ_Binary"].values
y_dev = dev_data["PHQ_Binary"].values
y_test = test_data["PHQ_Binary"].values

imputer = SimpleImputer(strategy="median").fit(X_train_raw)  # TRAIN ONLY
scaler = StandardScaler().fit(imputer.transform(X_train_raw))  # TRAIN ONLY

X_train = scaler.transform(imputer.transform(X_train_raw))
X_dev = scaler.transform(imputer.transform(X_dev_raw))
X_test = scaler.transform(imputer.transform(X_test_raw))

Train/Dev/Test: 163/56/56 | features: 161


In [4]:
"""
CELL 4 — TASK 3: Data/leakage audit (verification statements)
"""

print("=" * 70)
print("DATA + LEAKAGE AUDIT")
print("=" * 70)
print(f"[OK] Participant-level stats computed independently per participant "
      f"(compute_participant_stats operates on one participant's array at a time, no cross-participant info)")
print(f"[OK] No labels used in feature generation (compute_participant_stats never receives y)")
print(f"[OK] Imputer fitted on X_train_raw only ({X_train_raw.shape[0]} rows), "
      f"applied via .transform() to dev ({X_dev_raw.shape[0]}) and test ({X_test_raw.shape[0]})")
print(f"[OK] Scaler fitted on imputed TRAIN only, applied via .transform() to dev/test")
print(f"[OK] Official split sizes preserved: train={len(train_data)} (expected 163), "
      f"dev={len(dev_data)} (expected 56), test={len(test_data)} (expected 56)")
print(f"[OK] Class distributions match official split: "
      f"train={train_data.PHQ_Binary.value_counts().to_dict()} (expect 126/37), "
      f"dev={dev_data.PHQ_Binary.value_counts().to_dict()} (expect 44/12), "
      f"test={test_data.PHQ_Binary.value_counts().to_dict()} (expect 39/17)")
print(f"\n[FLAG FOR PRIOR EXPERIMENTS] SVM/LogReg/XGBoost hyperparameter search used "
      f"GridSearchCV with StratifiedKFold on (X_train, y_train) only — TEST and DEV "
      f"were not part of the CV loop. This is correct.")
print(f"\n[REQUIREMENT FOR NEXT EXPERIMENT] ANOVA F-score feature selection MUST be fit "
      f"inside each CV fold, not once on all 163 train participants beforehand. "
      f"Implemented below via sklearn Pipeline(SelectKBest -> classifier) passed directly "
      f"to GridSearchCV, which correctly re-fits SelectKBest on each fold's training portion only.")

DATA + LEAKAGE AUDIT
[OK] Participant-level stats computed independently per participant (compute_participant_stats operates on one participant's array at a time, no cross-participant info)
[OK] No labels used in feature generation (compute_participant_stats never receives y)
[OK] Imputer fitted on X_train_raw only (163 rows), applied via .transform() to dev (56) and test (56)
[OK] Scaler fitted on imputed TRAIN only, applied via .transform() to dev/test
[OK] Official split sizes preserved: train=163 (expected 163), dev=56 (expected 56), test=56 (expected 56)
[OK] Class distributions match official split: train={0: 126, 1: 37} (expect 126/37), dev={0: 44, 1: 12} (expect 44/12), test={0: 39, 1: 17} (expect 39/17)

[FLAG FOR PRIOR EXPERIMENTS] SVM/LogReg/XGBoost hyperparameter search used GridSearchCV with StratifiedKFold on (X_train, y_train) only — TEST and DEV were not part of the CV loop. This is correct.

[REQUIREMENT FOR NEXT EXPERIMENT] ANOVA F-score feature selection MUST be fit 

In [5]:
"""
CELL 5 — TASK 4 + 6: Re-evaluate SVM, LogReg, XGBoost at threshold=0.50
AND at the DEV-selected threshold. Refits with the previously-identified
best hyperparameters (no new tuning — reproducing prior results for
comparison, not re-selecting anything using TEST).
"""

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, average_precision_score, confusion_matrix)

RANDOM_STATE = 42

def full_eval(y_true, proba, threshold):
    preds = (proba >= threshold).astype(int)
    return {
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_true, preds),
        "Precision": precision_score(y_true, preds, zero_division=0),
        "Recall": recall_score(y_true, preds, zero_division=0),
        "F1": f1_score(y_true, preds, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, proba),
        "PR_AUC": average_precision_score(y_true, proba),
        "ConfusionMatrix": confusion_matrix(y_true, preds).tolist(),
    }

def dev_best_threshold(y_dev_, dev_proba_):
    best_t, best_f1 = 0.5, -1
    for t in np.arange(0.10, 0.91, 0.01):
        f1 = f1_score(y_dev_, (dev_proba_ >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1

results_task4 = []

# --- SVM (C=1, gamma=0.001, as previously selected) ---
svm_model = SVC(kernel="rbf", C=1, gamma=0.001, class_weight="balanced",
                 probability=True, random_state=RANDOM_STATE).fit(X_train, y_train)
svm_dev_proba = svm_model.predict_proba(X_dev)[:, 1]
svm_test_proba = svm_model.predict_proba(X_test)[:, 1]
svm_thresh, svm_dev_f1 = dev_best_threshold(y_dev, svm_dev_proba)
for name, t in [("threshold=0.50", 0.50), (f"DEV-selected (t={svm_thresh:.2f})", svm_thresh)]:
    results_task4.append({"Model": "SVM", "Setting": name, "Split": "DEV", **full_eval(y_dev, svm_dev_proba, t)})
    results_task4.append({"Model": "SVM", "Setting": name, "Split": "TEST", **full_eval(y_test, svm_test_proba, t)})

# --- Logistic Regression (C=0.01, no PCA, as previously selected) ---
lr_model = LogisticRegression(C=0.01, class_weight="balanced", max_iter=5000,
                               random_state=RANDOM_STATE).fit(X_train, y_train)
lr_dev_proba = lr_model.predict_proba(X_dev)[:, 1]
lr_test_proba = lr_model.predict_proba(X_test)[:, 1]
lr_thresh, lr_dev_f1 = dev_best_threshold(y_dev, lr_dev_proba)
for name, t in [("threshold=0.50", 0.50), (f"DEV-selected (t={lr_thresh:.2f})", lr_thresh)]:
    results_task4.append({"Model": "LogReg", "Setting": name, "Split": "DEV", **full_eval(y_dev, lr_dev_proba, t)})
    results_task4.append({"Model": "LogReg", "Setting": name, "Split": "TEST", **full_eval(y_test, lr_test_proba, t)})

# --- XGBoost (previously selected params, best_iteration=36) ---
n_pos, n_neg = (y_train == 1).sum(), (y_train == 0).sum()
xgb_model = XGBClassifier(max_depth=2, n_estimators=36, learning_rate=0.03,
                           subsample=0.8, colsample_bytree=0.6,
                           scale_pos_weight=n_neg / n_pos, random_state=RANDOM_STATE,
                           eval_metric="logloss", verbosity=0).fit(X_train, y_train)
xgb_dev_proba = xgb_model.predict_proba(X_dev)[:, 1]
xgb_test_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_thresh, xgb_dev_f1 = dev_best_threshold(y_dev, xgb_dev_proba)
for name, t in [("threshold=0.50", 0.50), (f"DEV-selected (t={xgb_thresh:.2f})", xgb_thresh)]:
    results_task4.append({"Model": "XGBoost", "Setting": name, "Split": "DEV", **full_eval(y_dev, xgb_dev_proba, t)})
    results_task4.append({"Model": "XGBoost", "Setting": name, "Split": "TEST", **full_eval(y_test, xgb_test_proba, t)})

results_task4_df = pd.DataFrame(results_task4)
print("=" * 100)
print("TASK 4 + 6: THRESHOLD=0.50 vs DEV-SELECTED THRESHOLD (all models)")
print("=" * 100)
print(results_task4_df.drop(columns=["ConfusionMatrix"]).round(4).to_string(index=False))
print("\nConfusion matrices:")
for _, r in results_task4_df.iterrows():
    print(f"  {r['Model']:8s} | {r['Setting']:28s} | {r['Split']:4s} -> {r['ConfusionMatrix']}")

print(f"\nNote: ROC_AUC and PR_AUC are identical between threshold=0.50 and DEV-selected "
      f"for the same (model, split) pair, as expected (threshold-independent metrics) — "
      f"visible above where each model/split pair repeats the same ROC_AUC/PR_AUC value.")

TASK 4 + 6: THRESHOLD=0.50 vs DEV-SELECTED THRESHOLD (all models)
  Model               Setting Split  Threshold  Accuracy  Precision  Recall     F1  ROC_AUC  PR_AUC
    SVM        threshold=0.50   DEV       0.50    0.7857     0.0000  0.0000 0.0000   0.5739  0.3163
    SVM        threshold=0.50  TEST       0.50    0.6964     0.0000  0.0000 0.0000   0.6621  0.4523
    SVM DEV-selected (t=0.10)   DEV       0.10    0.2143     0.2143  1.0000 0.3529   0.5739  0.3163
    SVM DEV-selected (t=0.10)  TEST       0.10    0.3036     0.3036  1.0000 0.4658   0.6621  0.4523
 LogReg        threshold=0.50   DEV       0.50    0.4643     0.1786  0.4167 0.2500   0.5038  0.3075
 LogReg        threshold=0.50  TEST       0.50    0.5893     0.4118  0.8235 0.5490   0.6908  0.4744
 LogReg DEV-selected (t=0.34)   DEV       0.34    0.3214     0.2400  1.0000 0.3871   0.5038  0.3075
 LogReg DEV-selected (t=0.34)  TEST       0.34    0.4464     0.3542  1.0000 0.5231   0.6908  0.4744
XGBoost        threshold=0.50   DE

In [6]:
"""
CELL 6 — TASK 5: Score distribution analysis (LogReg, SVM)
"""

def score_distribution(proba, y_true, label):
    pos_scores = proba[y_true == 1]
    neg_scores = proba[y_true == 0]
    print(f"\n{label}:")
    print(f"  Positive class (n={len(pos_scores)}): "
          f"min={pos_scores.min():.4f} max={pos_scores.max():.4f} "
          f"mean={pos_scores.mean():.4f} median={np.median(pos_scores):.4f}")
    print(f"  Negative class (n={len(neg_scores)}): "
          f"min={neg_scores.min():.4f} max={neg_scores.max():.4f} "
          f"mean={neg_scores.mean():.4f} median={np.median(neg_scores):.4f}")

print("=" * 70)
print("TASK 5: SCORE DISTRIBUTIONS")
print("=" * 70)
print("\n--- Logistic Regression ---")
score_distribution(lr_dev_proba, y_dev, "DEV")
score_distribution(lr_test_proba, y_test, "TEST")

print("\n--- SVM ---")
score_distribution(svm_dev_proba, y_dev, "DEV")
score_distribution(svm_test_proba, y_test, "TEST")

print("\n" + "-" * 70)
print("Predicted-positive counts at fixed thresholds:")
threshold_check = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]
rows = []
for t in threshold_check:
    rows.append({
        "Threshold": t,
        "LogReg_DEV_pos": (lr_dev_proba >= t).sum(), "LogReg_TEST_pos": (lr_test_proba >= t).sum(),
        "SVM_DEV_pos": (svm_dev_proba >= t).sum(), "SVM_TEST_pos": (svm_test_proba >= t).sum(),
    })
print(pd.DataFrame(rows).to_string(index=False))
print(f"\n(DEV has {len(y_dev)} total, {y_dev.sum()} positive; TEST has {len(y_test)} total, {y_test.sum()} positive)")

TASK 5: SCORE DISTRIBUTIONS

--- Logistic Regression ---

DEV:
  Positive class (n=12): min=0.3485 max=0.8153 mean=0.5136 median=0.4784
  Negative class (n=44): min=0.2085 max=0.7950 mean=0.4952 median=0.5100

TEST:
  Positive class (n=17): min=0.4590 max=0.8011 mean=0.5988 median=0.6189
  Negative class (n=39): min=0.2408 max=0.7550 mean=0.5022 median=0.5140

--- SVM ---

DEV:
  Positive class (n=12): min=0.2245 max=0.2315 mean=0.2285 median=0.2286
  Negative class (n=44): min=0.2228 max=0.2321 mean=0.2280 median=0.2280

TEST:
  Positive class (n=17): min=0.2278 max=0.2325 mean=0.2298 median=0.2292
  Negative class (n=39): min=0.2249 max=0.2320 mean=0.2286 median=0.2287

----------------------------------------------------------------------
Predicted-positive counts at fixed thresholds:
 Threshold  LogReg_DEV_pos  LogReg_TEST_pos  SVM_DEV_pos  SVM_TEST_pos
       0.1              56               56           56            56
       0.2              56               56           56   

In [7]:
"""
CELL 6b — Diagnose + fix: use SVM decision_function instead of predict_proba
for thresholding, since probability=True's Platt scaling has collapsed on
this small/imbalanced dataset (see score distribution above).
"""

def minmax_scores(raw_scores, fit_range=None):
    lo, hi = fit_range if fit_range else (raw_scores.min(), raw_scores.max())
    return np.clip((raw_scores - lo) / (hi - lo + 1e-9), 0, 1), (lo, hi)

svm_dev_margin = svm_model.decision_function(X_dev)
svm_test_margin = svm_model.decision_function(X_test)

svm_dev_score, dev_range = minmax_scores(svm_dev_margin)
svm_test_score, _ = minmax_scores(svm_test_margin, fit_range=dev_range)  # same scale, no test leakage into fitting the range

print("SVM decision_function score distribution (min-max normalized, NOT Platt-scaled):")
score_distribution(svm_dev_score, y_dev, "DEV")
score_distribution(svm_test_score, y_test, "TEST")

svm_margin_thresh, svm_margin_dev_f1 = dev_best_threshold(y_dev, svm_dev_score)
print(f"\nBest DEV threshold on decision_function scores: {svm_margin_thresh:.2f} (DEV F1={svm_margin_dev_f1:.4f})")

svm_margin_dev_metrics = full_eval(y_dev, svm_dev_score, svm_margin_thresh)
svm_margin_test_metrics = full_eval(y_test, svm_test_score, svm_margin_thresh)
print("\nSVM (decision_function-based) DEV :", {k: v for k, v in svm_margin_dev_metrics.items() if k != "ConfusionMatrix"})
print("SVM (decision_function-based) TEST:", {k: v for k, v in svm_margin_test_metrics.items() if k != "ConfusionMatrix"})
print("DEV  confusion:", svm_margin_dev_metrics["ConfusionMatrix"])
print("TEST confusion:", svm_margin_test_metrics["ConfusionMatrix"])

SVM decision_function score distribution (min-max normalized, NOT Platt-scaled):

DEV:
  Positive class (n=12): min=0.1853 max=0.9323 mean=0.6100 median=0.6227
  Negative class (n=44): min=0.0000 max=1.0000 mean=0.5605 median=0.5593

TEST:
  Positive class (n=17): min=0.5408 max=1.0000 mean=0.7538 median=0.6897
  Negative class (n=39): min=0.2257 max=0.9841 mean=0.6250 median=0.6387

Best DEV threshold on decision_function scores: 0.60 (DEV F1=0.4211)

SVM (decision_function-based) DEV : {'Threshold': np.float64(0.5999999999999998), 'Accuracy': 0.6071428571428571, 'Precision': 0.3076923076923077, 'Recall': 0.6666666666666666, 'F1': 0.42105263157894735, 'ROC_AUC': np.float64(0.5738636363636364), 'PR_AUC': np.float64(0.3162938414395129)}
SVM (decision_function-based) TEST: {'Threshold': np.float64(0.5999999999999998), 'Accuracy': 0.5892857142857143, 'Precision': 0.42105263157894735, 'Recall': 0.9411764705882353, 'F1': 0.5818181818181818, 'ROC_AUC': np.float64(0.6621417797888387), 'PR_AUC

In [8]:
"""
CELL 7 (corrected) — ANOVA F-score feature-selection sweep.
SVM branch uses decision_function (min-max scaled) instead of predict_proba
for DEV thresholding, per the diagnosed Platt-scaling collapse. LogReg and
XGBoost keep predict_proba (verified reliable in Task 5).
"""

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV

K_LIST = [10, 20, 30, 40, 50, 75, 100, 161]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

classifier_configs = {
    "XGBoost": (
        XGBClassifier(scale_pos_weight=n_neg / n_pos, random_state=RANDOM_STATE,
                       eval_metric="logloss", verbosity=0),
        {"clf__max_depth": [2, 3], "clf__n_estimators": [100, 200], "clf__learning_rate": [0.03, 0.05]},
        "predict_proba",
    ),
    "LogReg": (
        LogisticRegression(class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE),
        {"clf__C": [0.001, 0.01, 0.1, 1, 10]},
        "predict_proba",
    ),
    "SVM": (
        SVC(kernel="rbf", class_weight="balanced", probability=False, random_state=RANDOM_STATE),  # no Platt scaling needed
        {"clf__C": [0.1, 1, 10], "clf__gamma": ["scale", 0.01, 0.001]},
        "decision_function",
    ),
}

def get_scores(pipe, X, score_type, fit_range=None):
    """Returns (scores, range_used). For decision_function, min-max normalizes;
    fit_range=None fits the range on X itself (use only for DEV), otherwise
    reuses a previously-fit range (use for TEST, to avoid fitting on test)."""
    if score_type == "predict_proba":
        return pipe.predict_proba(X)[:, 1], None
    else:
        raw = pipe.decision_function(X)
        lo, hi = fit_range if fit_range else (raw.min(), raw.max())
        return np.clip((raw - lo) / (hi - lo + 1e-9), 0, 1), (lo, hi)

feature_selection_results = []
fitted_pipelines = {}

for clf_name, (base_clf, param_grid, score_type) in classifier_configs.items():
    print(f"\n{'='*70}\n{clf_name} ({score_type}) — sweeping K over {K_LIST}\n{'='*70}")
    for K in K_LIST:
        pipe = Pipeline([
            ("select", SelectKBest(f_classif, k=K)),
            ("clf", base_clf),
        ])
        search = GridSearchCV(pipe, param_grid, scoring="f1", cv=cv, n_jobs=-1)
        search.fit(X_train, y_train)

        best_pipe = search.best_estimator_
        fitted_pipelines[(clf_name, K)] = best_pipe

        dev_scores_k, dev_range_k = get_scores(best_pipe, X_dev, score_type)  # range fit on DEV itself
        dev_thresh_k, dev_f1_k = dev_best_threshold(y_dev, dev_scores_k)

        row = {
            "Model": clf_name, "K": K, "ScoreType": score_type,
            "CV_F1": search.best_score_, "Best_Params": search.best_params_,
            "DEV_Threshold": dev_thresh_k, "DEV_F1": dev_f1_k,
            "DEV_ScoreRange": dev_range_k,  # stored so TEST reuses this exact range, never test-derived
        }
        feature_selection_results.append(row)
        print(f"  K={K:3d} | CV F1={search.best_score_:.4f} | DEV F1={dev_f1_k:.4f} "
              f"(t={dev_thresh_k:.2f}) | params={search.best_params_}")

fs_results_df = pd.DataFrame(feature_selection_results)
fs_results_df.drop(columns=["Best_Params", "DEV_ScoreRange"]).to_csv(
    "/kaggle/working/feature_selection_sweep_results.csv", index=False)
print("\nFull K-sweep saved.")


XGBoost (predict_proba) — sweeping K over [10, 20, 30, 40, 50, 75, 100, 161]


/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [ 56  57  58  59  60  61  79  80  81  87  89  91 104 125 126 127 128 129
 130 133 135 137] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [ 56  57  58  59  60  61  79  80  81  87  89  91 104 125 126 127 128 129
 130 133 135 137] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
/usr/local/lib/python3.12/dist-packages/sklearn/feature_select

  K= 10 | CV F1=0.3190 | DEV F1=0.3438 (t=0.10) | params={'clf__learning_rate': 0.05, 'clf__max_depth': 2, 'clf__n_estimators': 100}


/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [ 56  57  58  59  60  61  79  80  81  87  89  91 104 125 126 127 128 129
 130 133 135 137] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [ 56  57  58  59  60  61  79  80  81  87  89  91 104 125 126 127 128 129
 130 133 135 137] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
/usr/local/lib/python3.12/dist-packages/sklearn/feature_select

  K= 20 | CV F1=0.3149 | DEV F1=0.3000 (t=0.33) | params={'clf__learning_rate': 0.03, 'clf__max_depth': 2, 'clf__n_estimators': 200}


...

/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [ 56  57  58  59  60  61  79  80  81  87  89  91 104 125 126 127 128 129
 130 133 135 137] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [ 56  57  58  59  60  61  79  80  81  87  89  91 104 125 126 127 128 129
 130 133 135 137] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
/usr/local/lib/python3.12/dist-packages/sklearn/feature_select

  K=100 | CV F1=0.3220 | DEV F1=0.4211 (t=0.64) | params={'clf__C': 1, 'clf__gamma': 0.001}


/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [ 56  57  58  59  60  61  79  80  81  87  89  91 104 125 126 127 128 129
 130 133 135 137] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [ 56  57  58  59  60  61  79  80  81  87  89  91 104 125 126 127 128 129
 130 133 135 137] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
/usr/local/lib/python3.12/dist-packages/sklearn/feature_select

  K=161 | CV F1=0.3182 | DEV F1=0.4211 (t=0.60) | params={'clf__C': 1, 'clf__gamma': 0.001}

Full K-sweep saved.


/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [ 56  57  58  59  60  61  79  80  81  87  89  91 104 125 126 127 128 129
 130 133 135 137] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [ 56  57  58  59  60  61  79  80  81  87  89  91 104 125 126 127 128 129
 130 133 135 137] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


In [9]:
"""
CELL 8 (corrected) — Final K selection + frozen TEST evaluation,
using the correct score type per classifier (decision_function for SVM,
predict_proba for LogReg/XGBoost). SVM's TEST score range reuses the
range fit on DEV (never fit on TEST) for consistent min-max scaling.
"""

final_results = []
confusion_table = []

for clf_name, (_, _, score_type) in classifier_configs.items():
    sub = fs_results_df[fs_results_df.Model == clf_name]
    best_row = sub.sort_values(["DEV_F1", "CV_F1"], ascending=False).iloc[0]
    best_K = int(best_row["K"])
    best_pipe = fitted_pipelines[(clf_name, best_K)]
    frozen_threshold = best_row["DEV_Threshold"]
    dev_range = best_row["DEV_ScoreRange"]

    dev_scores_final, _ = get_scores(best_pipe, X_dev, score_type)
    test_scores_final, _ = get_scores(best_pipe, X_test, score_type, fit_range=dev_range)  # reuse DEV's range

    for setting_name, t in [("DEV-selected", frozen_threshold), ("threshold=0.50", 0.50)]:
        dev_m = full_eval(y_dev, dev_scores_final, t)
        test_m = full_eval(y_test, test_scores_final, t)
        final_results.append({"Model": clf_name, "K": best_K, "Setting": setting_name,
                               "Split": "DEV", "CV_F1": best_row["CV_F1"], **dev_m})
        final_results.append({"Model": clf_name, "K": best_K, "Setting": setting_name,
                               "Split": "TEST", "CV_F1": best_row["CV_F1"], **test_m})
        confusion_table.append({"Model": clf_name, "K": best_K, "Setting": setting_name,
                                 "Split": "TEST", "ConfusionMatrix": test_m["ConfusionMatrix"]})

final_results_df = pd.DataFrame(final_results)
print("=" * 100)
print("FINAL FEATURE-SELECTION RESULTS")
print("=" * 100)
print(final_results_df.drop(columns=["ConfusionMatrix"]).round(4).to_string(index=False))

print("\nComparison table: Model | K | CV F1 | DEV F1 | TEST F1 | TEST ROC-AUC | TEST PR-AUC | TEST Accuracy")
comparison_rows = []
for clf_name in classifier_configs:
    dev_row = final_results_df[(final_results_df.Model == clf_name) & (final_results_df.Setting == "DEV-selected") & (final_results_df.Split == "DEV")].iloc[0]
    test_row = final_results_df[(final_results_df.Model == clf_name) & (final_results_df.Setting == "DEV-selected") & (final_results_df.Split == "TEST")].iloc[0]
    comparison_rows.append({
        "Model": clf_name, "K": dev_row["K"], "CV_F1": round(dev_row["CV_F1"], 4),
        "DEV_F1": round(dev_row["F1"], 4), "TEST_F1": round(test_row["F1"], 4),
        "TEST_ROC_AUC": round(test_row["ROC_AUC"], 4), "TEST_PR_AUC": round(test_row["PR_AUC"], 4),
        "TEST_Accuracy": round(test_row["Accuracy"], 4),
    })
comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))

print("\nConfusion-matrix table (TEST, DEV-selected threshold):")
for r in confusion_table:
    if r["Setting"] == "DEV-selected":
        print(f"  {r['Model']:8s} (K={r['K']:3d}) -> {r['ConfusionMatrix']}")

final_results_df.drop(columns=["ConfusionMatrix"]).to_csv("/kaggle/working/feature_selection_final_results.csv", index=False)
comparison_df.to_csv("/kaggle/working/feature_selection_comparison_table.csv", index=False)

FINAL FEATURE-SELECTION RESULTS
  Model  K        Setting Split  CV_F1  Threshold  Accuracy  Precision  Recall     F1  ROC_AUC  PR_AUC
XGBoost 50   DEV-selected   DEV 0.3633       0.12    0.2679     0.2264  1.0000 0.3692   0.3920  0.2378
XGBoost 50   DEV-selected  TEST 0.3633       0.12    0.3036     0.3036  1.0000 0.4658   0.6176  0.4441
XGBoost 50 threshold=0.50   DEV 0.3633       0.50    0.5000     0.1667  0.3333 0.2222   0.3920  0.2378
XGBoost 50 threshold=0.50  TEST 0.3633       0.50    0.6250     0.4286  0.7059 0.5333   0.6176  0.4441
 LogReg 50   DEV-selected   DEV 0.3444       0.72    0.8214     0.6667  0.3333 0.4444   0.5360  0.4305
 LogReg 50   DEV-selected  TEST 0.3444       0.72    0.6786     0.4545  0.2941 0.3571   0.6229  0.3928
 LogReg 50 threshold=0.50   DEV 0.3444       0.50    0.5000     0.1923  0.4167 0.2632   0.5360  0.4305
 LogReg 50 threshold=0.50  TEST 0.3444       0.50    0.6429     0.4444  0.7059 0.5455   0.6229  0.3928
    SVM 30   DEV-selected   DEV 0.3300   

In [10]:
"""
CELL 9 — Feature interpretability for the selected XGBoost feature-selection
model: which features SelectKBest kept + their XGBoost gain importance.
"""

xgb_best_K = int(fs_results_df[fs_results_df.Model == "XGBoost"]
                  .sort_values(["DEV_F1", "CV_F1"], ascending=False).iloc[0]["K"])
xgb_best_pipe = fitted_pipelines[("XGBoost", xgb_best_K)]

selector = xgb_best_pipe.named_steps["select"]
clf = xgb_best_pipe.named_steps["clf"]

selected_mask = selector.get_support()
selected_features = [f for f, keep in zip(feature_cols, selected_mask) if keep]
importances = clf.feature_importances_

imp_df = pd.DataFrame({"Feature": selected_features, "XGB_Gain_Importance": importances})
imp_df["Original_eGeMAPS_Feature"] = imp_df["Feature"].apply(lambda f: "_".join(f.split("_")[:-1]))
imp_df["Statistic"] = imp_df["Feature"].apply(lambda f: f.split("_")[-1])
imp_df = imp_df.sort_values("XGB_Gain_Importance", ascending=False).reset_index(drop=True)

print(f"XGBoost feature-selection model: K={xgb_best_K} features selected via ANOVA F-score")
print(f"\nTop 20 selected features by XGBoost gain importance:")
print(imp_df.head(20).to_string(index=False))
print("\nNote: these are features used by this statistical model on this dataset — "
      "not a claim of clinical meaning or causal relevance to depression.")

imp_df.to_csv("/kaggle/working/xgboost_selected_feature_importance.csv", index=False)

# ------------------------------------------------------------------
# Save the fitted pipelines for reuse (no retraining needed later)
# ------------------------------------------------------------------
import joblib
joblib.dump(xgb_best_pipe, "/kaggle/working/best_xgboost_feature_selection_pipeline.joblib")
for clf_name in classifier_configs:
    best_K_c = int(fs_results_df[fs_results_df.Model == clf_name]
                    .sort_values(["DEV_F1", "CV_F1"], ascending=False).iloc[0]["K"])
    joblib.dump(fitted_pipelines[(clf_name, best_K_c)],
                f"/kaggle/working/best_{clf_name.lower()}_feature_selection_pipeline.joblib")
print("\nFitted pipelines saved to /kaggle/working/ (.joblib files).")

XGBoost feature-selection model: K=50 features selected via ANOVA F-score

Top 20 selected features by XGBoost gain importance:
                        Feature  XGB_Gain_Importance   Original_eGeMAPS_Feature Statistic
              mfcc3_sma3_median             0.086523                 mfcc3_sma3    median
         slope500-1500_sma3_max             0.066348         slope500-1500_sma3       max
                 mfcc2_sma3_p75             0.061306                 mfcc2_sma3       p75
                 mfcc2_sma3_p25             0.057539                 mfcc2_sma3       p25
                 mfcc3_sma3_min             0.055595                 mfcc3_sma3       min
                 mfcc3_sma3_p75             0.051001                 mfcc3_sma3       p75
         F1frequency_sma3nz_p25             0.050765         F1frequency_sma3nz       p25
         slope500-1500_sma3_std             0.048657         slope500-1500_sma3       std
         F1bandwidth_sma3nz_p75             0.043936         F